# Домашнее задание: Alibaba Model Studio и вызов бесплатной VLM по `/chat/completions`

## Цель

Нужно самостоятельно пройти путь от регистрации в **Alibaba Cloud Model Studio** до успешного вызова мультимодальной модели через OpenAI-совместимый endpoint `/chat/completions`.[1][2]

Результат домашнего задания — рабочий минимальный клиент, который отправляет в модель **изображение + текстовый запрос** и получает содержательный ответ модели.[2]

## Что нужно освоить

После выполнения задания студент должен уметь:

- найти и активировать Model Studio;[1]
- получить API key и выбрать корректный региональный endpoint;[1]
- вызвать OpenAI-совместимый API через Python или `curl`;
- передать в `messages` мультимодальный запрос формата `image_url + text`;[2]
- понимать, какие модели в Model Studio подходят для visual understanding и какие из них доступны с бесплатной квотой для новых пользователей в Singapore region.[3][1]

## Постановка задачи

Нужно выбрать **любую бесплатную VLM-модель**, доступную в Alibaba Model Studio для новых пользователей, и сделать не менее **трёх** успешных запросов через `/chat/completions`.[3][4]

Один из запросов должен быть на **описание изображения**, один — на **извлечение структурированной информации**, и один — на **reasoning по изображению**.[2]

## Рекомендуемые бесплатные модели

Для новых пользователей в **Singapore region** Alibaba Model Studio предоставляет free quota на 90 дней после активации сервиса для ряда моделей.[1][5]

Ниже — хорошие варианты для этого ДЗ.

| Модель | Почему подходит | Что важно знать |
|---|---|---|
| `qwen3.5-plus` | Универсальная мультимодальная модель, поддерживает text/image/video input и OpenAI-compatible вызовы.[1][2] | Это самый безопасный выбор для первого запуска VLM в этом ДЗ.[2] |
| `qwen3.5-flash` | Более быстрая и более дешёвая мультимодальная модель для latency-sensitive сценариев.[1][2] | Подходит, если нужен быстрый отклик и простой пайплайн.[2] |
| `qwen3-vl-plus` | Специализированная VLM-линейка для visual understanding, object localization и multimodal agents.[2] | Хороший вариант, если хочется протестировать именно «классическую» VLM-семью Qwen-VL.[2] |
| `qwen3-vl-flash` | Более лёгкий и быстрый вариант Qwen3-VL.[2] | Удобен для экспериментов с визуальными задачами и reasoning.[2] |
| `qwen-vl-max` | Подходит для general visual tasks и фигурирует в документации как visual understanding model.[4][1] | Это модель предыдущего семейства; её можно взять для сравнения с более новыми моделями.[2] |
| `qwen-vl-plus` | Более дешёвый и быстрый вариант внутри Qwen2.5-VL family.[2] | Полезно сравнить с `qwen-vl-max` на простом VQA или OCR.[2] |

## Что именно нужно сделать

### Шаг 1. Зарегистрироваться и активировать сервис

1. Создайте или используйте существующий аккаунт Alibaba Cloud.
2. Откройте Model Studio и активируйте сервис.[1]
3. Выберите регион. Для нового пользователя наиболее полезен **Singapore region**, потому что именно там заявлена free quota для новых пользователей.[1][5]

### Шаг 2. Получить API key

Сгенерируйте API key для выбранного региона и сохраните его как переменную окружения `DASHSCOPE_API_KEY`.[2]

Пример для Linux/macOS:

```bash
export DASHSCOPE_API_KEY="sk-..."
```

Пример для Windows PowerShell:

```powershell
$env:DASHSCOPE_API_KEY="sk-..."
```

### Шаг 3. Выбрать endpoint

Model Studio поддерживает OpenAI-compatible API, но `base_url` зависит от региона.[1]

Основные варианты:

- Singapore: `https://dashscope-intl.aliyuncs.com/compatible-mode/v1`
- US (Virginia): `https://dashscope-us.aliyuncs.com/compatible-mode/v1`
- China (Beijing): `https://dashscope.aliyuncs.com/compatible-mode/v1`
- Hong Kong: `https://cn-hongkong.dashscope.aliyuncs.com/compatible-mode/v1`

Для этого ДЗ рекомендуется использовать именно **Singapore endpoint**.[1][2]

### Шаг 4. Сделать первый мультимодальный запрос

Нужно отправить в модель массив `content`, где один элемент — картинка, а второй — текстовый вопрос.[2]

Минимальный пример на Python:

```python
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

response = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
          # TODO
    ]
)

print(response.choices[0].message.content)
```

Этот формат соответствует официальной документации Alibaba Cloud по visual understanding и OpenAI-compatible вызовам.[2]

Пример через `curl`:

```bash
curl --location 'https://dashscope-intl.aliyuncs.com/compatible-mode/v1/chat/completions' \
--header "Authorization: Bearer $DASHSCOPE_API_KEY" \
--header 'Content-Type: application/json' \
--data '{
  "model": "qwen3.5-plus",
  "messages": [
    # TODO
  ]
}'
```

## Обязательные эксперименты

Нужно сделать **три разных сценария**.

### Эксперимент A. Image captioning / VQA

Пример запроса:

```text
Кратко опиши изображение и перечисли 5 ключевых объектов сцены.
```

### Эксперимент B. Structured extraction

Нужно заставить модель вернуть структурированный ответ по изображению или документу.[2]

Пример запроса:

```text
Извлеки информацию с изображения и верни JSON со следующими полями: main_objects, scene_type, visible_text, colors.
```

### Эксперимент C. Visual reasoning

Нужно задать вопрос, где модель не просто перечисляет объекты, а делает вывод по изображению.[2]

Примеры:

```text
Что происходило непосредственно перед этим моментом и что вероятнее всего произойдет дальше? Объясни гипотезу.
```

```text
Есть ли на изображении признаки, указывающие на погоду, время суток или тип места? Обоснуй.
```

## Дополнительный уровень

Дополнительно можно протестировать:

- несколько изображений в одном запросе; это поддерживается документацией Model Studio для visual understanding.[2]
- streaming response; он поддерживается OpenAI-compatible API для chat completions.
- thinking mode для подходящих моделей семейства `qwen3.5` и `qwen3-vl`.[2]

Пример идеи для бонуса:

```text
Сравни два изображения и объясни, чем отличаются сцены, освещение и действия объектов.
```

## Что сдать

Нужно прислать один архив или одну папку со следующим содержимым:

1. `README.md` с кратким описанием, какую модель вы выбрали и почему.
2. `requirements.txt` или список используемых библиотек.
3. Скрипт `run_vlm.py` или `run_vlm.ipynb`.
4. Файл `report.md` с результатами экспериментов.
5. При желании — `curl_examples.txt`.

## Требования к `report.md`

В отчёте обязательно должно быть:

- название выбранной модели;
- выбранный регион;
- какой endpoint использовался;
- какие именно запросы отправлялись;
- скриншоты или текст ответов модели;
- краткий анализ: где модель ответила хорошо, а где ошиблась или была слишком общей;
- оценка удобства Model Studio как платформы для быстрого старта.[1][2]

## Критерии оценивания

1. 8 баллов - основная часть задания
2. 2 балла - вы смогли настроить не только alibaba cloud, но нашли и задокументировали еще какого-то другого бесплатного (или условно-бесплатного) провайдера VLM моделей, а так же смогли его вызвать.

## Рекомендуемая структура решения

```text
project/
├── README.md
├── requirements.txt
├── run_vlm.py
├── report.md
└── assets/
    ├── test_image_1.jpg
    └── test_image_2.jpg
```

## Шаблон `report.md`

```markdown
# Отчёт по домашнему заданию: Alibaba Model Studio VLM

## 1. Выбранная модель
- Модель:
- Регион:
- Endpoint:
- Почему выбрана именно она:

## 2. Настройка окружения
- Версия Python:
- Библиотеки:
- Как задавался API key:

## 3. Эксперименты

### 3.1 Image captioning / VQA
**Запрос:**

**Ответ модели:**

**Комментарий:**

### 3.2 Structured extraction
**Запрос:**

**Ответ модели:**

**Комментарий:**

### 3.3 Visual reasoning
**Запрос:**

**Ответ модели:**

**Комментарий:**

## Минимальный `requirements.txt`

```text
openai>=1.0.0
```

## Подсказки

- Если получаете `401`, почти всегда проблема в API key или в том, что ключ создан не для того региона.
- Если получаете `429`, это может быть rate limit или исчерпание доступной квоты.
- Если модель не принимает картинку, проверьте, что вы передаёте `messages[].content` как список объектов, а не как одну строку.[2]
- Для первого запуска лучше брать `qwen3.5-plus`, потому что он явно документирован как мультимодальная модель с OpenAI-compatible вызовами.[1][2]
